following tut: https://docs.fastf1.dev/gen_modules/examples_gallery/results_strategy/plot_strategy.html#sphx-glr-gen-modules-examples-gallery-results-strategy-plot-strategy-py

In [1]:
from matplotlib import pyplot as plt

import fastf1
import fastf1.plotting

In [2]:
session = fastf1.get_session(2022, "Canadian", 'R')
session.load()
laps = session.laps

req         WARNING 	DEFAULT CACHE ENABLED! (8.15 GB) C:\Users\onehi\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Canadian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '55', '44', '63', '16', '31', '77', '24', '14', '18', '3', '5', '23', '10', '

In [3]:
laps.columns


Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')

In [4]:
drivers = session.drivers
print(drivers)

['1', '55', '44', '63', '16', '31', '77', '24', '14', '18', '3', '5', '23', '10', '4', '6', '20', '22', '47', '11']


In [5]:
drivers = [session.get_driver(driver)["Abbreviation"] for driver in drivers]
print(drivers)

['VER', 'SAI', 'HAM', 'RUS', 'LEC', 'OCO', 'BOT', 'ZHO', 'ALO', 'STR', 'RIC', 'VET', 'ALB', 'GAS', 'NOR', 'LAT', 'MAG', 'TSU', 'MSC', 'PER']


In [6]:
stints = laps[["Driver", "Stint", "Compound", "LapNumber"]]
stints = stints.groupby(["Driver", "Stint", "Compound"])
stints = stints.count().reset_index()

In [7]:
stints = stints.rename(columns={"LapNumber": "StintLength"})
print(stints)

   Driver  Stint Compound  StintLength
0     ALB    1.0   MEDIUM           18
1     ALB    2.0     HARD           30
2     ALB    3.0     HARD           22
3     ALO    1.0   MEDIUM           28
4     ALO    2.0     HARD           21
5     ALO    3.0   MEDIUM           21
6     BOT    1.0     HARD           49
7     BOT    2.0   MEDIUM           21
8     GAS    1.0   MEDIUM            5
9     GAS    2.0     HARD           31
10    GAS    3.0     HARD           34
11    HAM    1.0   MEDIUM            9
12    HAM    2.0     HARD           35
13    HAM    3.0     HARD           26
14    LAT    1.0   MEDIUM            9
15    LAT    2.0     HARD           28
16    LAT    3.0     HARD           33
17    LEC    1.0     HARD           41
18    LEC    2.0   MEDIUM           29
19    MAG    1.0   MEDIUM            7
20    MAG    2.0     HARD           63
21    MSC    1.0   MEDIUM           19
22    NOR    1.0     HARD           19
23    NOR    2.0     HARD           23
24    NOR    3.0   MEDIUM

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(5, 10))


unique_compounds = stints["Compound"].str.upper().unique()
unique_compounds = [c for c in unique_compounds if c != "NONE"]

legend_patches = []
for comp in unique_compounds:
    color = fastf1.plotting.get_compound_color(comp, session=session)
    legend_patches.append(
        mpatches.Patch(color=color, label=comp.capitalize())
    )

for driver in drivers:
    driver_stints = stints.loc[stints["Driver"] == driver]
    driver_stints = driver_stints[driver_stints["Compound"].str.upper() != 'NONE']

    previous_stint_end = 0
    for idx, row in driver_stints.iterrows():
        compound_color = fastf1.plotting.get_compound_color(row["Compound"],
                                                            session=session)
        plt.barh(
            y=driver,
            width=row["StintLength"],
            left=previous_stint_end,
            color=compound_color,
            edgecolor="black",
            fill=True
        )
        previous_stint_end += row["StintLength"]

plt.title("2022 Canadian Grand Prix Strategies (Placement From Top to Bottom)")
plt.xlabel("Lap Number")
plt.grid(False)
ax.invert_yaxis()

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

# Add legend
plt.legend(handles=legend_patches, title="Tyre Compounds", loc="upper right")

plt.tight_layout()
plt.show()
